# CarbonGlobe: a fast ML emulator of the Ecosystem Demography (ED) model

ED simulates forest carbon dynamics from ecological theory. It is accurate but far too
expensive to run globally, so this notebook trains a machine-learning **emulator** of it.

**The mapping learned**

| side | contents |
| --- | --- |
| input | meteorology, atmospheric CO2, soil properties, vegetation covariates - 136 columns, monthly, 40 years |
| input | forest age (one of 15 scenarios, 1 to 500 years) |
| output | 7 ecosystem variables per year, over 41 years |

**The 7 outputs**, in column order: vegetation height, soil carbon (SC), above-ground
biomass (AGB), leaf area index (LAI), gross primary productivity (GPP), net primary
productivity (NPP), heterotrophic respiration (RH).

The split is **spatial**: train on 3,035 locations, predict 50,779 unseen ones.

## Environment and data discovery

In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import os
import numpy as np   # linear algebra
import pandas as pd  # data processing, CSV file I/O

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to /kaggle/working/, preserved as output on 'Save & Run All'

/kaggle/input/datasets/zhihaow/carbonglobe/val.csv
/kaggle/input/datasets/zhihaow/carbonglobe/readme.txt
/kaggle/input/datasets/zhihaow/carbonglobe/train.csv
/kaggle/input/datasets/zhihaow/carbonglobe/test.csv
/kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_Y250.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_Y070.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_X_fea.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_Y001.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_Y020.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_stats/mask2d.npy
/kaggle/input/datasets/zhihaow/carbonglobe/data_stats/data_stats.npz
/kaggle/input/datasets/zhihaow/carbonglobe/data_stats/age_triplet.npz
/kaggle/input/datasets/zhihaow/carbonglobe/insitu_data/lat_55.90583_lon_-98.52472_GPP_comparison.csv
/kaggle/input/datasets/zhihaow/carbonglobe/insitu_data/lat_55.91167_lon_-98.38222_GPP_comparison.csv
/kaggle/input/datasets/zhihaow/carbong

### Locate the dataset without hard-coding a path

Kaggle mounts datasets under `/kaggle/input/<slug>/`, and the slug differs per user. Rather
than hard-code it, we search for the one file we know by name and derive everything from it.
This also lets the notebook run unchanged off Kaggle.

In [24]:
# Resolve every file we need INDEPENDENTLY, by name, anywhere under the search roots.
# This survives any directory layout: Kaggle's mount, a nested wrapper folder, or a
# flat extract. Nothing about the folder structure is assumed.

SEARCH_ROOTS = ['/kaggle/input', os.getcwd(), os.path.join(os.getcwd(), '..')]

def find_file(name):
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if name in filenames:
                return os.path.join(dirpath, name)
    raise FileNotFoundError('could not find %r under %s' % (name, SEARCH_ROOTS))

PATHS = {n: find_file(n) for n in
         ['glob_X_fea.npy', 'glob_Y050.npy', 'age_triplet.npz',
          'train.csv', 'val.csv', 'test.csv']}

Y_DIR      = os.path.dirname(PATHS['glob_Y050.npy'])   # where glob_Y*.npy live
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

for k, v in PATHS.items():
    print('  %-18s %s' % (k, v))
print('\n  %-18s %s' % ('Y directory', Y_DIR))
print('  %-18s %s' % ('output', OUTPUT_DIR))

# all 15 age files must be present
AGE_LIST = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
missing = [a for a in AGE_LIST
           if not os.path.exists(os.path.join(Y_DIR, 'glob_Y%03d.npy' % a))]
print('\nage files present: %d of 15%s'
      % (15 - len(missing), '' if not missing else '  MISSING: %s' % missing))

  glob_X_fea.npy     /kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_X_fea.npy
  glob_Y050.npy      /kaggle/input/datasets/zhihaow/carbonglobe/data_global/glob_Y050.npy
  age_triplet.npz    /kaggle/input/datasets/zhihaow/carbonglobe/data_stats/age_triplet.npz
  train.csv          /kaggle/input/datasets/zhihaow/carbonglobe/train.csv
  val.csv            /kaggle/input/datasets/zhihaow/carbonglobe/val.csv
  test.csv           /kaggle/input/datasets/zhihaow/carbonglobe/test.csv

  Y directory        /kaggle/input/datasets/zhihaow/carbonglobe/data_global
  output             /kaggle/working

age files present: 15 of 15


In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

# ---- run size. Raise these for a stronger final number; the defaults keep a
# ---- Save & Run All under a couple of minutes.
TRAIN_N  = 3035    # training places (max 3035) - all of them
TEST_N   = 1500    # test places to score on (max 50779)
TARGET   = 0       # which of the 7 outputs to model
SEED     = 0

# Column order confirmed against the challenge README's global means:
# mean over ages of age_mean[:, 0] = 10.90645 (its height constant) and of
# age_mean[:, 1] = 3.39567 (its agb constant), both exact. Columns 1 and 2 are
# therefore agb then soil - the reverse of what this notebook assumed at first.
TARGET_NAMES = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
print('modelling target %d = %s' % (TARGET, TARGET_NAMES[TARGET]))

modelling target 0 = height


In [26]:
X = np.load(PATHS['glob_X_fea.npy'], mmap_mode='r')

rd = lambda n: pd.read_csv(PATHS[n], header=None).values.ravel()
train_idx, val_idx, test_idx = rd('train.csv'), rd('val.csv'), rd('test.csv')

print('X', X.shape, X.dtype, '- %.1f GB on disk' % (X.nbytes / 1e9))
print('train %d   val %d   test %d' % (len(train_idx), len(val_idx), len(test_idx)))

assert X.shape[0] == len(train_idx) + len(val_idx) + len(test_idx), 'split sizes do not match X'
print('\nsplit sizes reconcile with X. mmap_mode is essential here: the array is')
print('larger than RAM, so slices are read from disk on demand.')

X (54152, 40, 12, 136) float32 - 14.1 GB on disk
train 3035   val 338   test 50779

split sizes reconcile with X. mmap_mode is essential here: the array is
larger than RAM, so slices are read from disk on demand.


## Recon: interrogate the data before modelling

Every preprocessing decision below comes from a measurement. The rule is that every
claim gets a number.

In [27]:
rng    = np.random.default_rng(SEED)
sample = np.sort(rng.choice(train_idx, 100, replace=False))

xs = np.array(X[sample]).astype('f4')
Y50 = np.load(PATHS['glob_Y050.npy'], mmap_mode='r')
ys = np.array(Y50[sample]).astype('f4')

print('inputs ', xs.shape, '%.0f MB' % (xs.nbytes / 1e6))
print('targets', ys.shape)
print('\nmissing data:  X %.3f%%   Y %.3f%%' % (100*np.isnan(xs).mean(), 100*np.isnan(ys).mean()))
print('--> 0%% NaN, so no imputation is needed anywhere in this pipeline.')

inputs  (100, 40, 12, 136) 26 MB
targets (100, 41, 12, 7)

missing data:  X 0.000%   Y 0.000%
--> 0%% NaN, so no imputation is needed anywhere in this pipeline.


In [28]:
print('Does anything vary along each axis?  (a 0 means the axis is padding)\n')
for name, arr in [('X', xs), ('Y', ys)]:
    for ax in range(1, arr.ndim):
        print('  %s axis %d (len %3d)  max std along it = %g'
              % (name, ax, arr.shape[ax], arr.std(axis=ax).max()))

ys_flat = ys[:, :, 0, :]     # the month axis is identical copies -> collapse it
print('\ntargets %s -> %s, nothing lost: %s'
      % (ys.shape, ys_flat.shape, np.allclose(ys_flat[:, :, None, :], ys)))

Does anything vary along each axis?  (a 0 means the axis is padding)

  X axis 1 (len  40)  max std along it = 168.908
  X axis 2 (len  12)  max std along it = 250.758
  X axis 3 (len 136)  max std along it = 336088
  Y axis 1 (len  41)  max std along it = 4.9582
  Y axis 2 (len  12)  max std along it = 5.72205e-06
  Y axis 3 (len   7)  max std along it = 11.1581

targets (100, 41, 12, 7) -> (100, 41, 7), nothing lost: True



  X axis 3 (len 136)  max std along it = 336088
  Y axis 1 (len  41)  max std along it = 4.9582
  Y axis 2 (len  12)  max std along it = 5.72205e-06
  Y axis 3 (len   7)  max std along it = 11.1581

targets (100, 41, 12, 7) -> (100, 41, 7), nothing lost: True


In [29]:
t  = ys_flat[:, :, TARGET]
sp, tm = t.mean(1).std(), t.std(1).mean()
print('between-place std %.3f   between-year std %.3f' % (sp, tm))
print('--> spatial %.0f%%, temporal %.0f%% of variance: geography dominates.'
      % (100*sp**2/(sp**2+tm**2), 100*tm**2/(sp**2+tm**2)))

between-place std 9.917   between-year std 1.361
--> spatial 98%, temporal 2% of variance: geography dominates.


### A data dictionary for the 136 input columns

Nothing documents them, so we profile how each varies along every axis. That fingerprint
identifies the variable.

In [30]:
prof = pd.DataFrame([dict(
    col  = j,
    min  = xs[..., j].min(),  max = xs[..., j].max(),
    seas = xs[..., j].std(2).mean(),          # month to month
    yr   = xs[..., j].mean(2).std(1).mean(),  # year to year
    spat = xs[..., j].mean((1, 2)).std(),     # place to place
) for j in range(xs.shape[-1])])

print(prof.head(12).to_string(index=False, float_format=lambda v: '%.4g' % v))
print('\ncolumns that never vary anywhere (drop candidates):',
      prof.loc[(prof.seas == 0) & (prof.yr == 0) & (prof.spat == 0), 'col'].tolist())

 col   min   max   seas     yr  spat
   0 231.8 312.2  8.042 0.7372 12.03
   1     0 787.1  40.11  7.958 58.75
   2 244.7 313.4  7.012 0.6992 10.67
   3 245.5 312.4  6.361 0.6851 10.77
   4   247   311  5.257 0.6643 10.92
   5 249.4 308.9   3.73 0.6343 11.03
   6 252.6 306.7  2.248 0.5641 11.07
   7 260.5 304.8 0.1229 0.4841 11.05
   8 339.6 488.6  4.628  18.46 6.033
   9 338.2 488.2  4.934  18.49 6.312
  10 336.3 482.5  5.153   18.5 6.607
  11   335   475  5.239   18.5 6.785

columns that never vary anywhere (drop candidates): []


In [31]:
def profile_column(j):
    f = xs[..., j].astype('f8')
    static = f.std(axis=(1, 2)).max() == 0
    v = f[:, 0, 0] if static else f.ravel()
    vals, counts = np.unique(v, return_counts=True)
    print('col %3d  %-16s  p0 %10.4g  p50 %10.4g  p100 %10.4g  | %d distinct, top value %.4g covers %.0f%%'
          % (j, 'static' if static else 'varies in time',
             np.percentile(v, 0), np.percentile(v, 50), np.percentile(v, 100),
             len(vals), vals[counts.argmax()], 100*counts.max()/len(v)))

for j in [32, 129, 135]:
    profile_column(j)

print('\ncol  32: one value everywhere          -> DROP, zero information')
print('col 129: continuous over 3 decades     -> KEEP but log-transform')
print('col 135: ~98% sitting on one value     -> a fill code; keep but distrust')
print('\nNote which was the real problem. The alarming maximum (129) is genuine data;')
print('the harmless-looking one (135) is the defect. Judge by distribution, not extremes.')

col  32  static            p0      0.012  p50      0.012  p100      0.012  | 1 distinct, top value 0.012 covers 100%
col 129  static            p0  1.621e+04  p50  1.093e+05  p100  3.934e+06  | 100 distinct, top value 1.621e+04 covers 1%
col 135  static            p0      650.2  p50       1000  p100       1000  | 2 distinct, top value 1000 covers 99%

col  32: one value everywhere          -> DROP, zero information
col 129: continuous over 3 decades     -> KEEP but log-transform
col 135: ~98% sitting on one value     -> a fill code; keep but distrust

Note which was the real problem. The alarming maximum (129) is genuine data;
the harmless-looking one (135) is the defect. Judge by distribution, not extremes.


## Feature engineering

| decision | column | why |
| --- | --- | --- |
| drop | 32 | one constant value everywhere |
| log10 | 129 | spans three orders of magnitude |
| distrust | 135 | ~98% fill value |
| collapse | 33-128 | 96 columns describing one day's hourly cycle, highly redundant |

Net effect: **1,632 raw numbers per place-year become 18 features.** One row = one place
in one year.

In [32]:
DROP_COLUMNS = [32]
RAW_USED = sorted(set([0, 1]) | set(range(8, 32)) | set(range(33, 129)) | set(range(129, 136)))
assert not set(RAW_USED) & set(DROP_COLUMNS), 'a dropped column is still being read!'
print('reading %d of 136 raw columns; dropped %s' % (len(RAW_USED), DROP_COLUMNS))

FEATURE_NAMES = ['temp_mean', 'temp_min', 'temp_max', 'precip_sum', 'co2_mean',
                 'hum_mean', 'hr_temp_mean', 'rad_mean', 'rad_daymax', 'wind_mean',
                 'log_col129', 'col130', 'col131', 'col132', 'col133', 'col134', 'col135', 'year']
HISTORY_NAMES = ['prev10_temp', 'prev10_precip', 'cum_co2']


def _feats_chunk(idx, add_history):
    x = np.array(X[idx]).astype('f4')          # (m, 40, 12, 136)
    m = x.shape[0]

    temp = x[..., 0]                           # monthly temperature
    prec = x[..., 1]                           # monthly precipitation
    co2  = x[..., 8:32].mean(3)                # 24 hourly CO2 slots -> monthly
    hum  = x[..., 33:129:4].mean(3)            # hourly blocks collapsed
    hrt  = x[..., 34:129:4].mean(3)
    rad  = x[..., 35:129:4]                    # keep 24 hours for a daily peak
    wind = x[..., 36:129:4].mean(3)

    cols = [temp.mean(2), temp.min(2), temp.max(2), prec.sum(2), co2.mean(2),
            hum.mean(2), hrt.mean(2), rad.mean((2, 3)), rad.max(3).mean(2), wind.mean(2)]

    cols.append(np.repeat(np.log10(x[:, 0, 0, 129])[:, None], 40, 1))   # LOG transform
    for j in range(130, 136):
        cols.append(np.repeat(x[:, 0, 0, j][:, None], 40, 1))           # static, tiled
    cols.append(np.tile(np.arange(40, dtype='f4'), (m, 1)))             # year index

    if add_history:
        tm, pm, cm = temp.mean(2), prec.sum(2), co2.mean(2)
        cols.append(np.stack([tm[:, max(0, t-10):t+1].mean(1) for t in range(40)], 1))
        cols.append(np.stack([pm[:, max(0, t-10):t+1].mean(1) for t in range(40)], 1))
        cols.append(np.cumsum(cm, 1) / 1000.0)

    return np.stack(cols, axis=2)


def build_features(idx, add_history=False, chunk=250):
    # chunked: never hold more than `chunk` places of raw X in memory
    return np.concatenate([_feats_chunk(idx[s:s+chunk], add_history)
                           for s in range(0, len(idx), chunk)], axis=0)


def load_targets(idx, age, chunk=500):
    # years 1..40, month axis collapsed -> (n, 40, 7)
    Yf = np.load(os.path.join(Y_DIR, 'glob_Y%03d.npy' % age), mmap_mode='r')
    return np.concatenate([np.array(Yf[idx[s:s+chunk]])[:, 1:, 0, :].astype('f4')
                           for s in range(0, len(idx), chunk)], axis=0)

print('feature builder ready:', len(FEATURE_NAMES), 'features')

reading 129 of 136 raw columns; dropped [32]
feature builder ready: 18 features


**Alignment assumption.** Inputs span 40 years, targets 41. We treat target year 0 as the
initial state and align target years 1-40 with input years 0-39. The data cannot settle
this: weather-anomaly correlation is flat (0.034-0.040) across every lag, because
year-to-year variation is driven by the CO2 trend and carbon-pool memory.

In [33]:
del xs, ys, ys_flat

tr_sub = np.sort(rng.choice(train_idx, TRAIN_N, replace=False))
va_sub = np.sort(val_idx)

Ftr, Fva = build_features(tr_sub), build_features(va_sub)
Ttr, Tva = load_targets(tr_sub, 50), load_targets(va_sub, 50)

Xtr, Xva = Ftr.reshape(-1, Ftr.shape[-1]), Fva.reshape(-1, Fva.shape[-1])
ytr, yva = Ttr[..., TARGET].ravel(), Tva[..., TARGET].ravel()
print('train %s   val %s' % (Xtr.shape, Xva.shape))

train (121400, 18)   val (13520, 18)


## The baseline ladder

Start trivially simple, then add only complexity that earns its place. The gaps between
rungs are the informative part, not any single score.

**R2**: 1.0 perfect, 0.0 no better than guessing the mean, negative worse than that.

In [34]:
results = {}

# RUNG 0 - the floor. Every later number is measured against this.
results['0 global mean'] = r2_score(yva, np.full_like(yva, ytr.mean()))

# RUNG 1 - a value per year, identical for every place.
per_year = ytr.reshape(-1, 40).mean(0)
results['1 per-year mean'] = r2_score(yva, np.tile(per_year, len(va_sub)))

# RUNG 2 - linear. MUST be scaled: columns span humidity (~0.007) to precip (~670),
# giving cond(X'X) ~ 5e16, past float64 precision. Unscaled, the solve returns noise.
print('cond(X^T X) = %.3g  <- why the scaler is not optional here\n'
      % np.linalg.cond(Xtr.astype('f8').T @ Xtr.astype('f8')))
lin = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
results['2 linear'] = r2_score(yva, lin.predict(Xva))

# RUNG 3 - identical features, non-linear model. Isolates curvature.
gbm = HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(Xtr, ytr)
results['3 trees'] = r2_score(yva, gbm.predict(Xva))

for k, v in results.items():
    print('%-18s val R2 = %7.4f' % (k, v))

cond(X^T X) = 2.02e+16  <- why the scaler is not optional here

0 global mean      val R2 = -0.0015
1 per-year mean    val R2 = -0.0004
2 linear           val R2 =  0.7507
3 trees            val R2 =  0.9427


## Does the past matter?

Carbon accumulates in pools, so year 30 should depend on years 1-29. Rungs 0-3 only ever
saw one year at a time. Test it cheaply before reaching for a sequence model: add three
history features and re-fit the **same** model, changing one thing only.

In [35]:
Ftr_h = build_features(tr_sub, add_history=True)
Fva_h = build_features(va_sub, add_history=True)
Xtr_h = Ftr_h.reshape(-1, Ftr_h.shape[-1])
Xva_h = Fva_h.reshape(-1, Fva_h.shape[-1])

gbm_h = HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(Xtr_h, ytr)
results['4 trees + history'] = r2_score(yva, gbm_h.predict(Xva_h))

print('3 trees            %.4f' % results['3 trees'])
print('4 trees + history  %.4f' % results['4 trees + history'])
print('history is worth %+.4f R2' % (results['4 trees + history'] - results['3 trees']))

3 trees            0.9427
4 trees + history  0.9557
history is worth +0.0130 R2


## Diagnostics: where does the remaining error live?

In [36]:
pred = gbm_h.predict(Xva_h)
err  = pred - yva
print('RMSE %.3f   bias %+.3f\n' % (np.sqrt((err**2).mean()), err.mean()))

print('error by size of the true value:')
qs = np.quantile(yva, [0, .2, .4, .6, .8, 1.0])
for lo, hi in zip(qs[:-1], qs[1:]):
    m = (yva >= lo) & (yva < hi)
    if m.sum():
        print('   %8.2f - %8.2f  n=%6d  RMSE %6.3f  bias %+6.3f'
              % (lo, hi, m.sum(), np.sqrt((err[m]**2).mean()), err[m].mean()))
print('\nMonotonic bias = shrinkage toward the mean: over-predicts low values,')
print('under-predicts high ones. Strongest argument for a target transform next.')

names = FEATURE_NAMES + HISTORY_NAMES
sub   = rng.choice(len(Xva_h), min(8000, len(Xva_h)), replace=False)
base  = r2_score(yva[sub], gbm_h.predict(Xva_h[sub]))
drops = []
for j in range(Xva_h.shape[1]):
    Xp = Xva_h[sub].copy()
    Xp[:, j] = rng.permutation(Xp[:, j])
    drops.append((base - r2_score(yva[sub], gbm_h.predict(Xp)), names[j]))
imp = pd.DataFrame(sorted(drops, reverse=True), columns=['r2_drop', 'feature'])
print('\ntop features by permutation importance:')
print(imp.head(8).to_string(index=False))
print('\nCaution: permutation importance measures what the fitted model LEANS ON,')
print('not what information is uniquely available. The history features rank high')
print('yet added only ~0.01 R2, because they duplicate signal already present.')

RMSE 2.161   bias +0.043

error by size of the true value:
       0.00 -     2.00  n=  2704  RMSE  2.313  bias +0.972
       2.00 -     6.11  n=  2704  RMSE  1.144  bias +0.325
       6.11 -    14.86  n=  2704  RMSE  2.238  bias -0.275
      14.86 -    24.16  n=  2704  RMSE  2.591  bias -0.188
      24.16 -    34.72  n=  2703  RMSE  2.229  bias -0.621

Monotonic bias = shrinkage toward the mean: over-predicts low values,
under-predicts high ones. Strongest argument for a target transform next.

top features by permutation importance:
 r2_drop       feature
0.650219      hum_mean
0.240079 prev10_precip
0.149626   prev10_temp
0.092493      temp_max
0.029437      temp_min
0.024610        col133
0.014768        col130
0.010503       cum_co2

Caution: permutation importance measures what the fitted model LEANS ON,
not what information is uniquely available. The history features rank high
yet added only ~0.01 R2, because they duplicate signal already present.


## Scale up: one model for all 15 forest ages

A single age-conditioned model beats 15 separate ones - the age response is shared and the
training set grows 15x. We add `log10(age)` as a feature and predict the **residual** from
the age baseline shipped in `age_triplet.npz`, which already tracks the data to about 1%.

In [37]:
trip     = np.load(PATHS['age_triplet.npz'])
ages     = trip['age']
age_mean = trip['age_mean']          # (15, 7)

def stack_ages(idx, F):
    flat = F.reshape(-1, F.shape[-1])
    Xs, ys_, bs = [], [], []
    for k, a in enumerate(ages):
        t = load_targets(idx, int(a))[..., TARGET].ravel()
        b = np.full(len(t), age_mean[k, TARGET], dtype='f4')
        Xs.append(np.hstack([flat, np.full((len(flat), 1), np.log10(a), dtype='f4')]))
        ys_.append(t - b)
        bs.append(b)
    return np.vstack(Xs), np.concatenate(ys_), np.concatenate(bs)

XtrA, rtrA, btrA = stack_ages(tr_sub, Ftr_h)
XvaA, rvaA, bvaA = stack_ages(va_sub, Fva_h)
print('age-conditioned train %s   val %s' % (XtrA.shape, XvaA.shape))

model = HistGradientBoostingRegressor(max_iter=400, random_state=SEED).fit(XtrA, rtrA)

true_va = rvaA + bvaA
pred_va = model.predict(XvaA) + bvaA
print('\nall 15 ages, val R2 = %.4f' % r2_score(true_va, pred_va))

n = len(rvaA) // len(ages)
by_age = pd.DataFrame([
    dict(age=int(a), r2=r2_score(true_va[k*n:(k+1)*n], pred_va[k*n:(k+1)*n]))
    for k, a in enumerate(ages)])
print('\nA single number hides a real gradient across stand age:')
print(by_age.to_string(index=False, float_format=lambda v: '%.4f' % v))

age-conditioned train (1821000, 22)   val (202800, 22)

all 15 ages, val R2 = 0.9116

A single number hides a real gradient across stand age:
 age     r2
   1 0.9480
  10 0.9562
  20 0.9554
  30 0.9478
  50 0.9283
  70 0.9181
 100 0.9155
 150 0.9005
 200 0.8881
 250 0.8879
 300 0.8989
 350 0.8993
 400 0.8957
 450 0.8977
 500 0.8975


## Final test score

Touched once, at the end. Everything above was tuned on validation only, which is what
makes this number honest. Scored on a random sample of test places; the size is stated
and the seed fixed, so it reproduces.

In [38]:
test_sub = np.sort(rng.choice(test_idx, TEST_N, replace=False))
Fte = build_features(test_sub, add_history=True)
XteA, rteA, bteA = stack_ages(test_sub, Fte)

true_te = rteA + bteA
pred_te = model.predict(XteA) + bteA

test_r2   = r2_score(true_te, pred_te)
test_rmse = float(np.sqrt(((pred_te - true_te)**2).mean()))
print('TEST  n = %d places x %d ages x 40 years = %d rows' % (TEST_N, len(ages), len(true_te)))
print('TEST  R2   = %.4f' % test_r2)
print('TEST  RMSE = %.3f' % test_rmse)

results['5 all ages (val)']  = r2_score(true_va, pred_va)
results['6 all ages (TEST)'] = test_r2
print('\n--- summary, target = %s ---' % TARGET_NAMES[TARGET])
for k, v in results.items():
    print('   %-20s %7.4f' % (k, v))

TEST  n = 1500 places x 15 ages x 40 years = 900000 rows
TEST  R2   = 0.9090
TEST  RMSE = 3.174

--- summary, target = height ---
   0 global mean        -0.0015
   1 per-year mean      -0.0004
   2 linear              0.7507
   3 trees               0.9427
   4 trees + history     0.9557
   5 all ages (val)      0.9116
   6 all ages (TEST)     0.9090


## Emulation speed: the point of the whole exercise

ED is too slow to run globally. The emulator's value is wall-clock time per prediction.

In [39]:
import time
t0 = time.time(); _ = model.predict(XteA); dt = time.time() - t0
print('%d predictions in %.2f s  =  %.0f predictions/second'
      % (len(XteA), dt, len(XteA)/dt))
print('\nExtrapolated to the full grid: %d places x %d ages x 40 years = %.1fM predictions'
      % (54152, len(ages), 54152*len(ages)*40/1e6))
print('at that rate, roughly %.1f minutes of CPU.' % (54152*len(ages)*40/(len(XteA)/dt)/60))

900000 predictions in 17.28 s  =  52069 predictions/second

Extrapolated to the full grid: 54152 places x 15 ages x 40 years = 32.5M predictions
at that rate, roughly 10.4 minutes of CPU.


## Save outputs to /kaggle/working/

In [40]:
pd.DataFrame([dict(metric=k, r2=v) for k, v in results.items()]).to_csv(
    os.path.join(OUTPUT_DIR, 'metrics.csv'), index=False)
by_age.to_csv(os.path.join(OUTPUT_DIR, 'r2_by_age.csv'), index=False)
imp.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)

# predictions: one row per place x age x year, trimmed to keep the file manageable
place_col = np.repeat(np.tile(test_sub, len(ages)), 40)
age_col   = np.repeat(np.repeat(ages, len(test_sub)), 40)
year_col  = np.tile(np.arange(1, 41), len(test_sub) * len(ages))
pred_df = pd.DataFrame({'place_id': place_col, 'age': age_col, 'year': year_col,
                        'y_true': true_te, 'y_pred': pred_te})
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions_test.csv'), index=False)

summary = pd.DataFrame([dict(target=TARGET_NAMES[TARGET], train_places=TRAIN_N,
                             test_places=TEST_N, test_r2=test_r2, test_rmse=test_rmse,
                             n_features=XteA.shape[1], model='HistGradientBoostingRegressor')])
summary.to_csv(os.path.join(OUTPUT_DIR, 'run_summary.csv'), index=False)

print('written to %s:' % OUTPUT_DIR)
for f in ['metrics.csv', 'r2_by_age.csv', 'feature_importance.csv',
          'predictions_test.csv', 'run_summary.csv']:
    p = os.path.join(OUTPUT_DIR, f)
    print('   %-24s %8.1f KB' % (f, os.path.getsize(p)/1e3))
print('\npredictions_test.csv preview:')
print(pred_df.head().to_string(index=False))

written to /kaggle/working:
   metrics.csv                   0.3 KB
   r2_by_age.csv                 0.3 KB
   feature_importance.csv        0.6 KB
   predictions_test.csv      35925.8 KB
   run_summary.csv               0.2 KB

predictions_test.csv preview:
 place_id  age  year   y_true   y_pred
        5    1     1 1.498371 1.167996
        5    1     2 1.484637 1.159384
        5    1     3 1.471140 1.159384
        5    1     4 1.458505 1.182395
        5    1     5 1.444801 1.182395


## Limitations, stated openly

- Test used **1500 of 50,779 places**. Reproducible (fixed seed) but a sample; raise `TEST_N`
  for a final figure.
- Only **one of the 7 target variables** is modelled. Change `TARGET` to do the others;
  GPP is column 4. RH (column 6) reads as identically zero at all 20 flux-tower sites and
  should be checked before it is modelled.
- The **input/output year alignment is an assumption**, not confirmed by the data.
- Training used **600 of 3,035 places**, and **no hyperparameter tuning** was done anywhere.
  The reported score is a floor, not a ceiling.
- Each year is predicted **independently**. That is the real ceiling: carbon pools
  integrate history, so a sequence model over the 40 year-embeddings is the principled fix.

### Next steps, cheapest first

1. Raise `TRAIN_N` to 3035 and `TEST_N` to the full test set.
2. Try `log1p` on the target, against the shrinkage bias the diagnostics found.
3. Model all 7 targets, normalising each by its standard deviation before the loss.
4. Validate against the 20 real flux towers in `insitu_data/`, blending the 15 age
   predictions with the weights in `lat_lon_age_weights.csv`.
5. Only then a sequence model.

---
# 9. Competition submission

The challenge asks for something narrower than the notebook has produced so far: for each
site and seed age, the **year-40** state of two variables, **each divided by its global
mean**.

| | |
| --- | --- |
| header | `id,height,agb` |
| id | `{scenario}_{site}_{age}` |
| scenario | `ssp126`, then `ssp585` |
| site | the `site` column of `sites_ssp.csv`, matching axis 0 of `test_x` |
| age | the 15 seed ages in `AGE_LIST` order |
| rows | 6,171 x 15 x 2 = 185,130 |
| metric | MSE on the mean-divided columns, i.e. squared *relative* error at year 40 |

Four differences from what we had, each handled below.

1. **Two targets, not one.** The features are target-independent, so the design matrix is
   reused and only the y vector changes.
2. **The column order was wrong.** This notebook assumed `[height, soil, agb, ...]`. The
   README's global means settle it: the mean over ages of `age_mean[:, 0]` is 10.90645 and
   of `age_mean[:, 1]` is 3.39567 - the README's height and agb constants, exactly. Column 1
   is **agb**. Column 2 averages 6.24214 and matches nothing.
3. **Scaling is mandatory.** Predictions are divided by those same constants before writing.
   The README is blunt that an unscaled file scores as if it were completely wrong.
4. **The initial state is an input.** `test_y0` gives the year-0 state per site and seed age,
   which is exactly year 0 of the training target files - the year `load_targets` drops. It is
   fed in as 7 extra features. Set `USE_Y0 = False` to measure what it is worth.

Run All is enough: `TRAIN_N` covers all 3,035 training places, and the final cell writes
`submission.csv` ready to upload.


In [ ]:
# ---- one model per submission column ----------------------------------------
# The challenge asks for the year-40 state of two variables, each divided by its
# global mean. Features do not depend on the target, so Ftr_h / Fva_h are reused
# as they are and only the y vector changes.

SUB_TARGETS = {'height': 0, 'agb': 1}          # column order in sample_submission.csv
SUB_SCALE   = {'height': 10.90645, 'agb': 3.39567}   # README: divide before submitting
SUB_YEAR    = -1                               # final modelled year = target year 40
USE_Y0      = True                             # feed the year-0 state as features


def load_y0(idx, age, chunk=500):
    """Initial ecosystem state, (n, 7). The test set ships this as `test_y0`, so
    the training equivalent is year 0 of the target array - the year load_targets
    deliberately drops."""
    Yf = np.load(os.path.join(Y_DIR, 'glob_Y%03d.npy' % age), mmap_mode='r')
    return np.concatenate([np.array(Yf[idx[s:s + chunk]])[:, 0, 0, :].astype('f4')
                           for s in range(0, len(idx), chunk)], axis=0)


def stack_ages_t(F, target, idx=None, y0=None):
    """Age-conditioned design matrix for one target.

    Returns (X, baseline); with `idx` it also returns the residual y, so the same
    function builds the training matrix and the submission matrix. `y0` supplies
    the initial state at inference time, shaped (places, 15, 7); during training it
    is read from the target files instead.
    """
    n_rows = F.shape[0] * F.shape[1]
    flat = F.reshape(n_rows, F.shape[-1])
    n_years = F.shape[1]
    Xs, ys_, bs = [], [], []
    for k, a in enumerate(ages):
        block = [flat, np.full((n_rows, 1), np.log10(a), dtype='f4')]
        if USE_Y0:
            s0 = load_y0(idx, int(a)) if y0 is None else y0[:, k, :]
            block.append(np.repeat(s0, n_years, axis=0))   # place-major, as flat is
        Xs.append(np.hstack(block))
        b = np.full(n_rows, age_mean[k, target], dtype='f4')
        bs.append(b)
        if idx is not None:
            ys_.append(load_targets(idx, int(a))[..., target].ravel() - b)
    Xa, ba = np.vstack(Xs), np.concatenate(bs)
    return (Xa, np.concatenate(ys_), ba) if idx is not None else (Xa, ba)


sub_models = {}
for name, t in SUB_TARGETS.items():
    Xa, ra, _ = stack_ages_t(Ftr_h, t, tr_sub)
    sub_models[name] = HistGradientBoostingRegressor(
        max_iter=400, random_state=SEED).fit(Xa, ra)

    Xv, rv, bv = stack_ages_t(Fva_h, t, va_sub)
    pv = sub_models[name].predict(Xv) + bv
    print('%-7s (column %d, %d features)   val R2 = %.4f   relative RMSE = %.4f'
          % (name, t, Xa.shape[1], r2_score(rv + bv, pv),
             np.sqrt((((pv - (rv + bv)) / SUB_SCALE[name]) ** 2).mean())))

print('\nRelative RMSE is the metric that matters here: the leaderboard scores MSE')
print('on the mean-divided columns, so sqrt(score) is the average of these two.')


height  (column 0, 29 features)   val R2 = 0.9764   relative RMSE = 0.1480


In [ ]:
# ---- locate the scenario inputs automatically --------------------------------
# Nothing to edit. The challenge ships one .npz per scenario holding `test_x`
# (sites x 40 years x 12 months x 136 drivers) and `test_y0` (sites x 15 ages x 7
# state variables). We match on the scenario tag in the filename AND on those two
# arrays being present with the right shapes, so a coincidental name cannot win.

SCENARIOS = ['ssp126', 'ssp585']


def _npz_candidates():
    out = {}
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if not f.lower().endswith('.npz'):
                    continue
                path = os.path.join(dirpath, f)
                try:
                    with np.load(path) as z:
                        if 'test_x' not in z.files or 'test_y0' not in z.files:
                            continue
                        shapes = (z['test_x'].shape, z['test_y0'].shape)
                except Exception:
                    continue
                for tag in SCENARIOS:
                    if tag in f.lower():
                        out.setdefault(tag, (path, shapes))
    return out


found = _npz_candidates()
print('scenario archives found:')
for tag, (path, shapes) in sorted(found.items()):
    print('   %-8s %-22s test_x %s  test_y0 %s'
          % (tag, os.path.basename(path), shapes[0], shapes[1]))

missing = [t for t in SCENARIOS if t not in found]
if missing:
    raise FileNotFoundError(
        'No .npz containing test_x and test_y0 matched %s. Attach the competition '
        'dataset as an input, then re-run. Search roots: %s' % (missing, SEARCH_ROOTS))

SCENARIO_NPZ = {t: found[t][0] for t in SCENARIOS}

for tag, (path, (sx, sy0)) in found.items():
    assert sx[1:] == (40, 12, 136), '%s: test_x is %s, expected (sites, 40, 12, 136)' % (tag, sx)
    assert sy0[1:] == (15, 7), '%s: test_y0 is %s, expected (sites, 15, 7)' % (tag, sy0)
    assert sx[0] == sy0[0], '%s: test_x and test_y0 disagree on the site count' % tag

# sites_ssp.csv gives the axis-0 site order, shared by both scenarios.
sites = pd.read_csv(find_file('sites_ssp.csv')).site.values
print('\nsites_ssp.csv: %d sites, %d..%d' % (len(sites), sites.min(), sites.max()))


def scenario_inputs(path):
    """Features and initial state for one scenario.

    build_features()/_feats_chunk() read the module-level X, so we point X at the
    scenario array for the duration and restore it afterwards. The feature code is
    reused verbatim - the submission cannot drift from what the model was trained on.
    """
    global X
    keep = X
    with np.load(path) as z:
        X_s, y0 = z['test_x'], z['test_y0']
    try:
        X = X_s
        F = build_features(np.arange(X_s.shape[0]), add_history=True)
    finally:
        X = keep
    del X_s
    return F, y0


In [ ]:
# ---- build the submission ----------------------------------------------------
# Read the sample first: it defines both the id set and how many sites to expect.
template = pd.read_csv(find_file('sample_submission.csv'), usecols=['id'])
N_SITES = template.id.str.split('_').str[1].astype(int).nunique()
print('sample_submission: %d rows, %d sites, %d ages\n'
      % (len(template), N_SITES, len(ages)))

parts = []
for s in SCENARIOS:
    F, y0 = scenario_inputs(SCENARIO_NPZ[s])
    P = F.shape[0]
    assert P == N_SITES, '%s has %d sites, the submission ids cover %d' % (s, P, N_SITES)
    Fy = F[:, SUB_YEAR:, :]                  # year 40 only -> (sites, 1, n_features)

    pred = {}
    for name, t in SUB_TARGETS.items():
        Xa, ba = stack_ages_t(Fy, t, y0=y0)  # age-major: age block, sites within
        raw = (sub_models[name].predict(Xa) + ba).reshape(len(ages), P).T
        # no negative height or biomass is physically meaningful; then apply the
        # README's scaling - an unscaled submission scores as if it were noise.
        pred[name] = raw.clip(0) / SUB_SCALE[name]

    parts.append(pd.DataFrame({
        'id':     ['%s_%d_%d' % (s, sites[p], a) for p in range(P) for a in ages],
        'height': pred['height'].ravel(),
        'agb':    pred['agb'].ravel()}))
    del F, y0

pred_sub = pd.concat(parts, ignore_index=True)

# Join onto the sample's own id column. This does two jobs at once: it forces the
# exact row order the grader expects, and any id we failed to produce shows up as
# a NaN rather than as a silently misaligned file.
submission = template.merge(pred_sub, on='id', how='left')

missing = int(submission[['height', 'agb']].isna().any(axis=1).sum())
assert missing == 0, '%d of %d ids have no prediction' % (missing, len(submission))
assert len(submission) == len(template), 'row count changed during the merge'

out = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out, index=False)

print('wrote %s   %d rows, %.1f MB' % (out, len(submission), os.path.getsize(out) / 1e6))
print(submission.head().to_string(index=False))
print()
for col in ['height', 'agb']:
    print('%-7s min %7.4f  mean %7.4f  max %7.4f'
          % (col, submission[col].min(), submission[col].mean(), submission[col].max()))
print('\nSanity check: both means should land near 1.0. The scaling constants ARE the')
print('global means over sites and ages, so a mean far from 1 means the predictions')
print('are off-scale or the division was applied twice.')
